# Stage 2 BBO capstone, Round 1 (Module 12)

Eight hidden functions. Each takes a few numbers between 0 and 1 and returns a single number, and I want that number as high as possible. I only get one query per function this round, so each one has to be chosen rather than guessed.

The approach is standard Bayesian optimisation: fit a Gaussian process to the data I already have, ask it for both a prediction and an uncertainty at any new point, then use Expected Improvement to trade those two off against each other and pick where to look next.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from scipy.stats import norm

np.random.seed(0)   # same answers every run

DATA = Path.cwd().parent / "data"   # notebook sits in notebooks/
LO, HI = 0.0, 0.999999                      # portal only accepts inputs in this range
DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}

def load(i):
    X = np.load(DATA / f"function_{i}" / "inputs.npy")
    y = np.load(DATA / f"function_{i}" / "outputs.npy")
    return X, y

print("ready")

ready


## What I am starting with

Checking the shapes against the FAQ before anything else, because if the files are wrong then everything downstream is wasted.

In [2]:
summary = pd.DataFrame([
    {"function": i, "dims": load(i)[0].shape[1], "points": len(load(i)[1]),
     "lowest y": round(load(i)[1].min(), 4), "highest y": round(load(i)[1].max(), 4)}
    for i in range(1, 9)
])
summary

,function,dims,points,lowest y,highest y
0,1,2,10,-0.0036,0.0000
1,2,2,10,-0.0656,0.6112
2,3,3,15,-0.3989,-0.0348
3,4,4,30,-32.6257,-4.0255
4,5,4,20,0.1129,1088.8596
5,6,5,20,-2.5712,-0.7143
6,7,6,30,0.0027,1.3650
7,8,8,40,5.5922,9.5985


The sizes match the FAQ, so the data is right.

The thing worth noticing is how far apart the output scales are. Function 5 runs into the thousands while Function 1 is so small it rounds to zero at four decimal places. No single model is going to cope with both, so Functions 1 and 5 get a log transform before anything is fitted to them. For Function 1 that also means dropping the negative readings, since I cannot take the log of those.

## The model

A Gaussian process, because it gives me a prediction and an honest uncertainty at every point, and the uncertainty is what tells me where exploring might pay off.

Three choices inside it:

- **Matern rather than RBF.** RBF assumes the function is perfectly smooth, which makes the model very confident in places it has no data. Matern with `nu=2.5` allows a rougher function and is the more cautious bet.
- **One length scale per input.** This lets the model work out for itself which inputs barely affect the output, which matters a lot on Function 8 where I have 8 inputs and only 40 points.
- **Noise.** Function 2 is the only one the brief describes as noisy, so it is the only one given real room in the `WhiteKernel`.

In [3]:
def prepare(i, X, y):
    """f1 and f5 span too many orders of magnitude to model directly, so log them."""
    if i == 1:
        keep = y > 0                      # cannot log the negative readings
        return X[keep], np.log(y[keep])
    if i == 5:
        return X, np.log(y)
    return X, y


def fit_gp(X, y, noisy=False):
    noise = WhiteKernel(0.01, (1e-5, 1.0)) if noisy else WhiteKernel(1e-6, (1e-10, 1.0))
    kernel = ConstantKernel(1.0) * Matern(np.full(X.shape[1], 0.3), (0.01, 10), nu=2.5) + noise
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                  n_restarts_optimizer=10, random_state=0)
    gp.fit(X, y)
    return gp

print(fit_gp(*load(4)).kernel_)

4.11**2 * Matern(length_scale=[1.83, 1.62, 1.84, 1.78], nu=2.5) + WhiteKernel(noise_level=1e-10)


## Choosing where to look next

Expected Improvement asks a single question: if I sample here, how much better than my current best do I expect to do? It handles exploration against exploitation on its own, because a candidate can score well either by having a high predicted value or by having a lot of uncertainty around it.

To find where EI is highest I score 50,000 random candidate points and keep the best. The course notes list random search as a perfectly acceptable way to do this and it avoids writing anything more complicated.

Function 1 is the exception. Its readings are so close to zero everywhere that the model sees a flat surface, so a global EI search would just return whichever corner has the widest error bars. Instead I search only within 0.05 of its strongest reading, which turns the query into a deliberate local probe rather than a meaningless one.

In [4]:
def expected_improvement(mu, sd, best, xi=0.01):
    sd = np.maximum(sd, 1e-12)
    z = (mu - best - xi) / sd
    return (mu - best - xi) * norm.cdf(z) + sd * norm.pdf(z)


np.random.seed(0)   # so re-running just this cell reproduces the same queries
rows = []
for i in range(1, 9):
    X, y = load(i)
    X_use, y_use = prepare(i, X, y)
    gp = fit_gp(X_use, y_use, noisy=(i == 2))

    if i == 1:
        # f1 reads as zero almost everywhere, so probe close to its strongest point
        centre = X_use[y_use.argmax()]
        cand = np.clip(centre + np.random.uniform(-0.05, 0.05, (50000, 2)), LO, HI)
    else:
        cand = np.random.uniform(LO, HI, (50000, DIMS[i]))

    mu, sd = gp.predict(cand, return_std=True)
    ei = expected_improvement(mu, sd, y_use.max())
    k = ei.argmax()

    rows.append({
        "function": i,
        "query": "-".join(f"{v:.6f}" for v in cand[k]),
        # did uncertainty or the prediction win this pick?
        "driver": "exploration" if sd[k] > np.median(sd) else "exploitation",
    })

queries = pd.DataFrame(rows)
queries

,function,query,driver
0,1,0.780987-0.726384,exploration
1,2,0.698303-0.001390,exploitation
2,3,0.474926-0.997260-0.403326,exploration
3,4,0.450103-0.409276-0.338614-0.419619,exploitation
4,5,0.020928-0.587748-0.857028-0.999232,exploration
5,6,0.405259-0.134616-0.965755-0.994201-0.011769,exploration
6,7,0.062044-0.626399-0.436325-0.174001-0.365819-0...,exploitation
7,8,0.031633-0.000480-0.195204-0.085573-0.932309-0...,exploration


## Queries for submission

Four picks came out as exploitation and four as exploration, which is a reasonable split for a first round on data this thin. Saving them in portal format, one per line.

In [5]:
out = Path.cwd().parent / "submissions" / "round1_module12.txt"
out.parent.mkdir(exist_ok=True)
out.write_text("\n".join(f"Function {r.function}: {r.query}" for r in queries.itertuples()))
print(out.read_text())

Function 1: 0.780987-0.726384
Function 2: 0.698303-0.001390
Function 3: 0.474926-0.997260-0.403326
Function 4: 0.450103-0.409276-0.338614-0.419619
Function 5: 0.020928-0.587748-0.857028-0.999232
Function 6: 0.405259-0.134616-0.965755-0.994201-0.011769
Function 7: 0.062044-0.626399-0.436325-0.174001-0.365819-0.743300
Function 8: 0.031633-0.000480-0.195204-0.085573-0.932309-0.341013-0.008251-0.520783


## What I take into Round 2

Function 1 is the one I expect to have learned least from. Every positive reading it has given so far is zero to within floating point error, so there is no gradient for the model to follow and this round's query is really a local probe rather than an informed guess. If it comes back near zero again, the sensible next step is to fit the shape the brief implies, a signal that decays with distance from a single source, rather than continuing to ask a Gaussian process to model a flat surface.

The higher dimensional functions, 7 and 8, are the other place to be careful. Forty points spread over eight inputs is very thin cover, so I should expect the model's confident looking peaks there to be partly invented, and I should keep leaning towards exploration until the data catches up.